# UCI HAR — 1D CNN + LSTM

Trains and evaluates the human-activity classifier on the UCI HAR inertial signals.

The code lives in `src/jepasafe/demopart.py`; this notebook imports it so there is
only ever one copy to fix. Architecture:

```
(B, 128, 9)
  Conv1d(9->64, k=5) + BN + ReLU
  Conv1d(64->64, k=5) + BN + ReLU
  MaxPool1d(2) + Dropout(0.3)     -> (B, 64, 60)
  LSTM(64), final hidden state    -> (B, 64)
  Dropout + Linear(64) + ReLU + Linear(6)
```

> This is a PC-side accuracy benchmark, not an FPGA deployment candidate —
> 61k parameters against the Tang Nano 9K's 8,640 LUT4 budget, and hls4ml's
> LSTM support is thin. The deployable model is in `src/jepasafe/model.py`.

## Setup

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start=None):
    """Walk up until CLAUDE.md turns up. Notebooks have no reliable __file__,
    so this keys off the working directory instead."""
    start = Path(start or Path.cwd()).resolve()
    for directory in [start, *start.parents]:
        if (directory / "CLAUDE.md").is_file():
            return directory
    raise FileNotFoundError(f"no CLAUDE.md above {start} - is the notebook inside the repo?")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from jepasafe.demopart import CLASS_NAMES, DEFAULT_ROOT, run

print("repo root :", REPO_ROOT)
print("dataset   :", DEFAULT_ROOT)
print("classes   :", CLASS_NAMES)

## Train

`run()` returns a dict: the trained `model`, per-epoch `history`, `y_true` /
`y_pred`, and the metrics. Nothing needs re-running to plot below.

First call parses the raw `.txt` files (~2 s) and caches `.npy` under
`data/processed/uci_har/`; later calls load the cache.

In [ ]:
result = run(epochs=30)

## Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cm = result["confusion_matrix"]
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"UCI HAR - CNN+LSTM ({result['accuracy']:.2f}% accuracy)")

# Counts in the cells, recoloured where the fill goes dark
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, f"{cm[i, j]}",
            ha="center", va="center", fontsize=9,
            color="white" if cm_norm[i, j] > 0.5 else "black",
        )

fig.colorbar(im, ax=ax, label="fraction of true class")
fig.tight_layout()
plt.show()

The usual UCI HAR failure mode is the SITTING / STANDING block: both are
stationary, so the accelerometer sees almost the same thing and only device
orientation separates them. The three walking classes are easy by comparison.

## Training curves

In [ ]:
history = result["history"]
epochs = range(1, len(history["loss"]) + 1)

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))

ax_loss.plot(epochs, history["loss"], label="train")
ax_loss.plot(epochs, history["val_loss"], label="validation")
ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("cross-entropy loss")
ax_loss.set_title("Loss"); ax_loss.legend(); ax_loss.grid(alpha=0.3)

ax_acc.plot(epochs, history["accuracy"], label="train")
ax_acc.plot(epochs, history["val_accuracy"], label="validation")
ax_acc.axvline(result["best_epoch"], ls="--", c="grey", lw=1,
               label=f"best (epoch {result['best_epoch']})")
ax_acc.set_xlabel("epoch"); ax_acc.set_ylabel("accuracy")
ax_acc.set_title("Accuracy"); ax_acc.legend(); ax_acc.grid(alpha=0.3)

fig.tight_layout()
plt.show()

Validation loss turning up while training loss keeps falling is overfitting.
`run()` restores the best-validation-accuracy weights before scoring, which is
why the reported number can beat the final epoch.

Note that "validation" here **is** the test split, so the restored-best number is
selected on the data it is reported on. State that in the write-up, or hold out a
real validation split from the training subjects if the figure needs to be strict.

## Per-class breakdown

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    result["y_true"], result["y_pred"], target_names=CLASS_NAMES, digits=4
))

## Optional: ablation against the plain LSTM

Same data, same optimizer, same metrics — only the CNN front end differs.
This is the number that justifies the convolutions in the report.

In [ ]:
baseline = run(model="lstm", epochs=30, verbose=False)

print(f"{'model':<12}{'accuracy':>10}{'F1':>10}")
print(f"{'LSTM':<12}{baseline['accuracy']:>9.2f}%{baseline['f1']:>10.4f}")
print(f"{'CNN+LSTM':<12}{result['accuracy']:>9.2f}%{result['f1']:>10.4f}")
print(f"{'delta':<12}{result['accuracy'] - baseline['accuracy']:>+9.2f}pp"
      f"{result['f1'] - baseline['f1']:>+10.4f}")